# AU FINANCE BANK

In [43]:
import pandas as pd
import xml.etree.ElementTree as ET
import pdfplumber
# Au

In [44]:

def to_page_zero(pdf):
    page = pdf.pages[0]

    table = page.extract_tables()[4]

    header = []

    for i in table[0]:
        header.append(i.replace('\n', ' '))
    
    table[0] = header
    
    table[1][2] = table[1][2].replace('\n', ' ')

    for i in range(1, len(table[0]) - 1):
        table[i][2] = table[i][2].replace('\n', ' ')
        table[i][3] = table[i][3].replace('\n', '')
    
    df = pd.DataFrame(table)
    return df
    

def to_page_all(page):
    
        table = page.extract_tables()[0]

        header = []

        for i in table[0]:
            header.append(i.replace('\n', ' '))
        
        table[0] = header

        table[0][2] = table[0][2].replace('\n', ' ')


        for i in range(1, len(table[0]) - 1):
            table[i][2] = table[i][2].replace('\n', ' ')
            table[i][3] = table[i][3].replace('\n', '')
        
        return pd.DataFrame(table)


In [45]:
# Converter functions

def convert_to_xlsx(df, path: str, sheet_name: str, index: bool = False):
    df.to_excel(path, sheet_name=sheet_name, index=index)


def clean_num(val):
    try:
        v = str(val).replace(',', '').strip()
        if v in ['', '-', 'nan', 'None']:
            return 0.0
        return float(v)
    except:
        return 0.0


# def df_to_tally_xml(df, output_path, bank_ledger="AU Small Finance Bank"):

#     root = ET.Element("ENVELOPE")

#     header = ET.SubElement(root, "HEADER")
#     ET.SubElement(header, "TALLYREQUEST").text = "Import Data"

#     body = ET.SubElement(root, "BODY")
#     importdata = ET.SubElement(body, "IMPORTDATA")

#     requestdesc = ET.SubElement(importdata, "REQUESTDESC")
#     ET.SubElement(requestdesc, "REPORTNAME").text = "Vouchers"

#     requestdata = ET.SubElement(importdata, "REQUESTDATA")

#     count = 0

#     for _, row in df.iterrows():
#         try:
#             dr = clean_num(row["Withdrawal (Dr)"])
#             cr = clean_num(row["Deposit (Cr)"])

#             if dr == 0 and cr == 0:
#                 continue

#             is_payment = dr > 0
#             vch_type = "Payment" if is_payment else "Receipt"
#             amount = dr if is_payment else cr

#             # Date format
#             date = pd.to_datetime(row["Date"], errors="coerce")
#             if pd.isna(date):
#                 continue
#             date_str = date.strftime("%Y%m%d")

#             narration = str(row.get("Narration", "Bank Entry"))
#             ref = str(row.get("Ref", ""))

#             if ref not in ['', 'nan', '-', '0']:
#                 narration += f" | Ref: {ref}"

#             # -------- TALLY MESSAGE --------
#             tallymessage = ET.SubElement(
#                 requestdata,
#                 "TALLYMESSAGE",
#                 {"xmlns:UDF": "TallyUDF"}
#             )

#             voucher = ET.SubElement(
#                 tallymessage,
#                 "VOUCHER",
#                 {
#                     "VCHTYPE": vch_type,
#                     "ACTION": "Create",
#                     "OBJVIEW": "Accounting Voucher View"
#                 }
#             )

#             # -------- BASIC FIELDS --------
#             ET.SubElement(voucher, "DATE").text = date_str
#             ET.SubElement(voucher, "VOUCHERTYPENAME").text = vch_type
#             ET.SubElement(voucher, "EFFECTIVEDATE").text = date_str
#             ET.SubElement(voucher, "NARRATION").text = narration
#             ET.SubElement(voucher, "PERSISTEDVIEW").text = "Accounting Voucher View"
#             ET.SubElement(voucher, "ISOPTIONAL").text = "No"
#             ET.SubElement(voucher, "ISDELETED").text = "No"
#             ET.SubElement(voucher, "HASCASHFLOW").text = "Yes"

#             # -------- PARTY LEDGER --------
#             party_name = "Suspense"
#             ET.SubElement(voucher, "PARTYLEDGERNAME").text = party_name

#             # -------- PARTY ENTRY --------
#             p_list = ET.SubElement(voucher, "ALLLEDGERENTRIES.LIST")
#             ET.SubElement(p_list, "LEDGERNAME").text = party_name
#             ET.SubElement(p_list, "ISDEEMEDPOSITIVE").text = "Yes" if is_payment else "No"
#             ET.SubElement(p_list, "AMOUNT").text = f"{-amount:.2f}" if is_payment else f"{amount:.2f}"

#             # -------- BANK ENTRY --------
#             b_list = ET.SubElement(voucher, "ALLLEDGERENTRIES.LIST")
#             ET.SubElement(b_list, "LEDGERNAME").text = bank_ledger
#             ET.SubElement(b_list, "ISDEEMEDPOSITIVE").text = "No" if is_payment else "Yes"
#             ET.SubElement(b_list, "AMOUNT").text = f"{amount:.2f}" if is_payment else f"{-amount:.2f}"

#             # -------- BANK ALLOCATION --------
#             bank_alloc = ET.SubElement(b_list, "BANKALLOCATIONS.LIST")
#             ET.SubElement(bank_alloc, "DATE").text = date_str
#             ET.SubElement(bank_alloc, "INSTRUMENTDATE").text = date_str
#             ET.SubElement(bank_alloc, "TRANSACTIONTYPE").text = "Others"
#             ET.SubElement(bank_alloc, "PAYMENTMODE").text = "Transacted"
#             ET.SubElement(bank_alloc, "AMOUNT").text = f"{amount:.2f}"

#             count += 1

#         except Exception as e:
#             print("Row error:", e)
#             continue

#     # -------- SAVE FILE --------
#     tree = ET.ElementTree(root)
#     tree.write(output_path, encoding="utf-8", xml_declaration=True)

#     print(f"✅ SUCCESS: {count} vouchers created → {output_path}")



In [46]:
def au_processor(path):
    
    with pdfplumber.open(path) as pdf:
        df = to_page_zero(pdf)

        for i in range(1, len(pdf.pages)):
            page = pdf.pages[i]
            frame = to_page_all(page)
            df = pd.concat([df, frame])
        
        return df

In [47]:

frame = au_processor('../temp/au.pdf')
# convert_to_xlsx(frame, '../output/au.xlsx', 'sheet1', index=False)


In [ ]:
def au_to_tally_vouchers(df, path: str, name: str, bank_ledger_name):

    df.columns = [str(col).strip() for col in df.columns]

    col_date = next((c for c in df.columns if 'transaction date' in c.lower()), None)
    col_narr = next((c for c in df.columns if 'description' in c.lower()), None)
    col_ref = next((c for c in df.columns if 'ref' in c.lower()), None)
    col_dr = next((c for c in df.columns if 'debit' in c.lower()), None)
    col_cr = next((c for c in df.columns if 'credit' in c.lower()), None)

    root = ET.Element("ENVELOPE")

    # HEADER
    header = ET.SubElement(root, "HEADER")
    ET.SubElement(header, "TALLYREQUEST").text = "Import Data"

    body = ET.SubElement(root, "BODY")
    importdata = ET.SubElement(body, "IMPORTDATA")

    requestdesc = ET.SubElement(importdata, "REQUESTDESC")
    ET.SubElement(requestdesc, "REPORTNAME").text = "All Masters"
    ET.SubElement(requestdesc, "STATICVARIABLES")

    requestdata = ET.SubElement(importdata, "REQUESTDATA")

    # 🔹 LEDGER CREATION (EXACT LIKE YOUR SAMPLE)
    msg = ET.SubElement(requestdata, "TALLYMESSAGE", {"xmlns:UDF": "TallyUDF"})

    ledger = ET.SubElement(msg, "LEDGER", {"NAME": "Suspense", "RESERVEDNAME": ""})
    ET.SubElement(ledger, "CURRENCYNAME").text = "₹"
    ET.SubElement(ledger, "PARENT").text = "Suspense A/c"
    ET.SubElement(ledger, "TAXTYPE").text = "Others"
    ET.SubElement(ledger, "ISBILLWISEON").text = "No"
    ET.SubElement(ledger, "ISCOSTCENTRESON").text = "No"
    ET.SubElement(ledger, "ISDELETED").text = "No"

    count = 0

    for _, row in df.iterrows():
        try:
            dr = clean_num(row[col_dr])
            cr = clean_num(row[col_cr])

            if dr == 0 and cr == 0:
                continue

            is_payment = dr > 0
            amount = dr if dr > 0 else cr
            date = pd.to_datetime(row[col_date]).strftime('%Y%m%d')

            narration = str(row[col_narr]).replace("\n", " ")
            ref = str(row[col_ref])

            msg = ET.SubElement(requestdata, "TALLYMESSAGE", {"xmlns:UDF": "TallyUDF"})

            vch = ET.SubElement(msg, "VOUCHER", {
                "VCHTYPE": "Payment",
                "ACTION": "Create",
                "OBJVIEW": "Accounting Voucher View"
            })

            # BASIC
            ET.SubElement(vch, "DATE").text = date
            ET.SubElement(vch, "VOUCHERTYPENAME").text = "Payment"
            ET.SubElement(vch, "NARRATION").text = narration
            ET.SubElement(vch, "PARTYLEDGERNAME").text = bank_ledger_name
            ET.SubElement(vch, "PERSISTEDVIEW").text = "Accounting Voucher View"
            ET.SubElement(vch, "EFFECTIVEDATE").text = date
            ET.SubElement(vch, "ISOPTIONAL").text = "No"
            ET.SubElement(vch, "ISDELETED").text = "No"
            ET.SubElement(vch, "HASCASHFLOW").text = "Yes"

            # 🔹 ENTRY 1 (Suspense)
            l1 = ET.SubElement(vch, "ALLLEDGERENTRIES.LIST")
            ET.SubElement(l1, "LEDGERNAME").text = "Suspense"
            ET.SubElement(l1, "ISDEEMEDPOSITIVE").text = "Yes"
            ET.SubElement(l1, "AMOUNT").text = f"-{amount:.2f}"

            # 🔹 ENTRY 2 (Bank)
            l2 = ET.SubElement(vch, "ALLLEDGERENTRIES.LIST")
            ET.SubElement(l2, "LEDGERNAME").text = bank_ledger_name
            ET.SubElement(l2, "ISDEEMEDPOSITIVE").text = "No"
            ET.SubElement(l2, "AMOUNT").text = f"{amount:.2f}"

            # 🔹 FULL BANK ALLOCATION (MATCHED)
            bank = ET.SubElement(l2, "BANKALLOCATIONS.LIST")
            ET.SubElement(bank, "DATE").text = date
            ET.SubElement(bank, "INSTRUMENTDATE").text = date
            ET.SubElement(bank, "TRANSACTIONTYPE").text = "Cheque"
            ET.SubElement(bank, "PAYMENTFAVOURING").text = "Suspense"
            ET.SubElement(bank, "INSTRUMENTNUMBER").text = ref
            ET.SubElement(bank, "PAYMENTMODE").text = "Transacted"
            ET.SubElement(bank, "BANKPARTYNAME").text = "Suspense"
            ET.SubElement(bank, "AMOUNT").text = f"{amount:.2f}"

            count += 1

        except Exception as e:
            print("Error:", e)

    tree = ET.ElementTree(root)
    tree.write(f"{path}/{name}", encoding="utf-8", xml_declaration=True)

    print(f"✅ Created {count} vouchers (Tally Exact Format)")

In [49]:
au_to_tally_vouchers(frame, '../output', 'AU Bank Statement.xml', "AU Small Finance Bank-4463")

Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Error: None
Erro

In [63]:
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime

In [64]:
COLUMN_MAPPING = {
    "date": ["Date", "DATE", "Txn Date", "Value Date"],
    "narration": ["Narration", "Description", "Remarks"],
    "debit": ["Debit", "Withdrawal"],
    "credit": ["Credit", "Deposit"],
    "amount": ["Amount"],
    "instrument": ["InstrumentNo", "Ref No", "Transaction ID"]
}

def find_column(df, possible_names):
    for col in df.columns:
        for name in possible_names:
            if col.strip().lower() == name.lower():
                return col
    return None

In [ ]:
def format_date(date):
    date = pd.to_datetime(date, errors="coerce")
    if pd.isna(date):
        return ""
    return date.strftime("%Y%m%d")

In [74]:
def clean_amount(val):
    if pd.isna(val):
        return 0

    val = str(val).strip()

    # Skip header or invalid values
    if val in ["", "-", "Debit (₹)", "Credit (₹)"]:
        return 0

    try:
        return float(val.replace(",", ""))
    except:
        return 0

In [66]:
def get_amount(row, debit_col, credit_col, amount_col):
    if amount_col:
        val = row.get(amount_col)
        if pd.notna(val):
            return float(val)

    debit = row.get(debit_col) if debit_col else None
    credit = row.get(credit_col) if credit_col else None

    if pd.notna(debit) and debit != 0:
        return float(debit)
    elif pd.notna(credit) and credit != 0:
        return float(credit)

    return 0

In [67]:
def create_voucher(row, party_ledger_name,
                   date_col, narration_col,
                   debit_col, credit_col, amount_col, instrument_col):

    voucher = ET.Element("TALLYMESSAGE", xmlns_UDF="TallyUDF")

    voucher_tag = ET.SubElement(
        voucher,
        "VOUCHER",
        VCHTYPE="Payment",
        ACTION="Create",
        OBJVIEW="Accounting Voucher View"
    )

    # Extract values
    date_val = format_date(row[date_col])
    narration_val = str(row.get(narration_col, "")).replace("\n", " ")
    amount = get_amount(row, debit_col, credit_col, amount_col)
    instrument = str(row.get(instrument_col, ""))

    # Basic Fields
    ET.SubElement(voucher_tag, "DATE").text = date_val
    ET.SubElement(voucher_tag, "VOUCHERTYPENAME").text = "Payment"
    ET.SubElement(voucher_tag, "NARRATION").text = narration_val
    ET.SubElement(voucher_tag, "PARTYLEDGERNAME").text = party_ledger_name
    ET.SubElement(voucher_tag, "PERSISTEDVIEW").text = "Accounting Voucher View"
    ET.SubElement(voucher_tag, "ISOPTIONAL").text = "No"
    ET.SubElement(voucher_tag, "EFFECTIVEDATE").text = date_val

    # =========================
    # 🧾 LEDGER ENTRIES
    # =========================

    # Debit (Expense / Suspense)
    ledger1 = ET.SubElement(voucher_tag, "ALLLEDGERENTRIES.LIST")
    ET.SubElement(ledger1, "LEDGERNAME").text = "Suspense"
    ET.SubElement(ledger1, "ISDEEMEDPOSITIVE").text = "Yes"
    ET.SubElement(ledger1, "AMOUNT").text = f"-{amount}"

    # Credit (Bank)
    ledger2 = ET.SubElement(voucher_tag, "ALLLEDGERENTRIES.LIST")
    ET.SubElement(ledger2, "LEDGERNAME").text = party_ledger_name
    ET.SubElement(ledger2, "ISDEEMEDPOSITIVE").text = "No"
    ET.SubElement(ledger2, "AMOUNT").text = f"{amount}"

    bank_alloc = ET.SubElement(ledger2, "BANKALLOCATIONS.LIST")
    ET.SubElement(bank_alloc, "DATE").text = date_val
    ET.SubElement(bank_alloc, "INSTRUMENTDATE").text = date_val
    ET.SubElement(bank_alloc, "TRANSACTIONTYPE").text = "Cheque"
    ET.SubElement(bank_alloc, "PAYMENTFAVOURING").text = "Suspense"
    ET.SubElement(bank_alloc, "INSTRUMENTNUMBER").text = instrument
    ET.SubElement(bank_alloc, "PAYMENTMODE").text = "Transacted"
    ET.SubElement(bank_alloc, "AMOUNT").text = f"{amount}"

    return voucher

In [75]:
def convert_excel_to_tally_xml(file_path, party_ledger_name, output_file="output.xml"):
    
    # 🔥 FIX: header=1
    df = pd.read_excel(file_path, header=1)

    df["Debit (₹)"] = pd.to_numeric(df["Debit (₹)"], errors="coerce")
    df["Credit (₹)"] = pd.to_numeric(df["Credit (₹)"], errors="coerce")

    df = df.fillna(0)

    df = df[df["Transaction Date"] != "Transaction Date"]

    print("✅ Columns:", df.columns.tolist())

    envelope = ET.Element("ENVELOPE")

    header = ET.SubElement(envelope, "HEADER")
    ET.SubElement(header, "TALLYREQUEST").text = "Import Data"

    body = ET.SubElement(envelope, "BODY")
    importdata = ET.SubElement(body, "IMPORTDATA")

    requestdesc = ET.SubElement(importdata, "REQUESTDESC")
    ET.SubElement(requestdesc, "REPORTNAME").text = "All Masters"

    requestdata = ET.SubElement(importdata, "REQUESTDATA")

    for _, row in df.iterrows():

        date_val = format_date(row["Transaction Date"])
        narration = str(row["Description/Narration"]).replace("\n", " ")

        debit = clean_amount(row["Debit (₹)"])
        credit = clean_amount(row["Credit (₹)"])

        amount = debit if debit > 0 else credit

        if amount == 0:
            continue

        instrument = str(row["Cheque/ Reference No."])

        voucher = ET.Element("TALLYMESSAGE", xmlns_UDF="TallyUDF")

        v = ET.SubElement(
            voucher,
            "VOUCHER",
            VCHTYPE="Payment",
            ACTION="Create",
            OBJVIEW="Accounting Voucher View"
        )

        ET.SubElement(v, "DATE").text = date_val
        ET.SubElement(v, "VOUCHERTYPENAME").text = "Payment"
        ET.SubElement(v, "NARRATION").text = narration
        ET.SubElement(v, "PARTYLEDGERNAME").text = party_ledger_name
        ET.SubElement(v, "EFFECTIVEDATE").text = date_val

        # Debit (Suspense)
        l1 = ET.SubElement(v, "ALLLEDGERENTRIES.LIST")
        ET.SubElement(l1, "LEDGERNAME").text = "Suspense"
        ET.SubElement(l1, "ISDEEMEDPOSITIVE").text = "Yes"
        ET.SubElement(l1, "AMOUNT").text = f"-{amount}"

        # Credit (Bank)
        l2 = ET.SubElement(v, "ALLLEDGERENTRIES.LIST")
        ET.SubElement(l2, "LEDGERNAME").text = party_ledger_name
        ET.SubElement(l2, "ISDEEMEDPOSITIVE").text = "No"
        ET.SubElement(l2, "AMOUNT").text = f"{amount}"

        bank = ET.SubElement(l2, "BANKALLOCATIONS.LIST")
        ET.SubElement(bank, "DATE").text = date_val
        ET.SubElement(bank, "INSTRUMENTDATE").text = date_val
        ET.SubElement(bank, "TRANSACTIONTYPE").text = "Cheque"
        ET.SubElement(bank, "PAYMENTFAVOURING").text = "Suspense"
        ET.SubElement(bank, "INSTRUMENTNUMBER").text = instrument
        ET.SubElement(bank, "PAYMENTMODE").text = "Transacted"
        ET.SubElement(bank, "AMOUNT").text = f"{amount}"

        requestdata.append(voucher)

    tree = ET.ElementTree(envelope)
    tree.write(output_file, encoding="utf-8", xml_declaration=True)

    print(f"✅ XML Generated: {output_file}")

In [76]:
convert_excel_to_tally_xml('../output/au.xlsx', "AU Small Finance Bank-4463")

✅ Columns: ['Transaction Date', 'Value Date', 'Description/Narration', 'Cheque/ Reference No.', 'Debit (₹)', 'Credit (₹)', 'Balance (₹)']
✅ XML Generated: output.xml
